# Road Accident Risk — Feature Engineering

## Objective

Extend the existing Toronto intersection-day dataset with
leakage-safe calendar, historical KSI, traffic, and weather features.

Evaluate their contribution using temporal validation
and a controlled ablation study.

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/intersection_day_2018_w5_features.csv",
    parse_dates=["prediction_date", "target_date"]
)

print(f"Dataset shape: {df.shape}")

Dataset shape: (823217, 8)


In [3]:
from pathlib import Path

sources = [
    Path("../data/interim/ksi_events.csv"),
    Path("../data/processed/ksi_matching_report.csv")
]

for path in sources:
    sample = pd.read_csv(path, nrows=2)

    print(f"\n{path.name}")
    print(sample.columns.tolist())


ksi_events.csv
['collision_id', 'accdate', 'stname1', 'stname2', 'stname3', 'acclass', 'accloc', 'traffictl', 'impactype', 'visible', 'light', 'rdsfcond', 'road_class', 'longitude', 'latitude', 'division', 'neighbourhood']

ksi_matching_report.csv
['match_status', 'event_count']


In [4]:
matched_events = pd.read_csv(
    "../data/processed/ksi_matched_events.csv",
    usecols=["intersection_id", "collision_id", "event_date"],
    parse_dates=["event_date"]
)

daily_ksi = (
    matched_events
    .groupby(["intersection_id", "event_date"])["collision_id"]
    .nunique()
    .reset_index(name="daily_ksi")
)

print(f"Matched KSI events: {len(matched_events):,}")
print(f"Intersection-days with KSI: {len(daily_ksi):,}")

Matched KSI events: 1,706
Intersection-days with KSI: 1,703


In [5]:
history_start = (
    df["prediction_date"].min()
    - pd.Timedelta(days=365)
)

history_dates = pd.date_range(
    start=history_start,
    end=df["prediction_date"].max(),
    freq="D"
)

history_grid = pd.MultiIndex.from_product(
    [df["intersection_id"].unique(), history_dates],
    names=["intersection_id", "prediction_date"]
).to_frame(index=False)

history_grid = (
    history_grid.merge(
        daily_ksi.rename(
            columns={"event_date": "prediction_date"}
        ),
        on=["intersection_id", "prediction_date"],
        how="left"
    )
    .sort_values(["intersection_id", "prediction_date"])
)

history_grid["daily_ksi"] = (
    history_grid["daily_ksi"].fillna(0)
)

for window in [30, 365]:
    history_grid[f"ksi_count_{window}d"] = (
        history_grid.groupby("intersection_id")["daily_ksi"]
        .transform(
            lambda s: s.shift(1)
            .rolling(window, min_periods=1)
            .sum()
            .fillna(0)
        )
        .astype("int32")
    )

print("History grid shape:", history_grid.shape)

History grid shape: (1648340, 5)


In [6]:
history_features = history_grid[
    [
        "intersection_id",
        "prediction_date",
        "ksi_count_30d",
        "ksi_count_365d"
    ]
]

df = df.merge(
    history_features,
    on=["intersection_id", "prediction_date"],
    how="left",
    validate="one_to_one"
)

assert (
    len(df) == 823_217
    and df[["ksi_count_30d", "ksi_count_365d"]].notna().all().all()
)

print("Updated dataset shape:", df.shape)

Updated dataset shape: (823217, 10)


In [7]:
# Calendar feature
df["is_weekend"] = (
    df["target_date"]
    .dt.dayofweek
    .isin([5, 6])
    .astype("int8")
)

# Traffic missingness indicator
df["traffic_missing"] = (
    df["historical_traffic_volume"]
    .isna()
    .astype("int8")
)

print("Dataset shape:", df.shape)
print("Traffic missing:", df["traffic_missing"].sum())

Dataset shape: (823217, 12)
Traffic missing: 325206


In [11]:
from pathlib import Path

weather_paths = [
    Path("../data/raw/en_climate_daily_ON_6158731_2017_P1D.csv"),
    Path("../data/raw/en_climate_daily_ON_6158731_2018_P1D.csv")
]

weather_parts = [
    pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    for path in weather_paths
]

weather_raw = pd.concat(
    weather_parts,
    ignore_index=True
)

print("Weather shape:", weather_raw.shape)
print("\nWeather columns:")
print(weather_raw.columns.tolist())

Weather shape: (730, 31)

Weather columns:
['Longitude (x)', 'Latitude (y)', 'Station Name', 'Climate ID', 'Date/Time', 'Year', 'Month', 'Day', 'Data Quality', 'Max Temp (°C)', 'Max Temp Flag', 'Min Temp (°C)', 'Min Temp Flag', 'Mean Temp (°C)', 'Mean Temp Flag', 'Heat Deg Days (°C)', 'Heat Deg Days Flag', 'Cool Deg Days (°C)', 'Cool Deg Days Flag', 'Total Rain (mm)', 'Total Rain Flag', 'Total Snow (cm)', 'Total Snow Flag', 'Total Precip (mm)', 'Total Precip Flag', 'Snow on Grnd (cm)', 'Snow on Grnd Flag', 'Dir of Max Gust (10s deg)', 'Dir of Max Gust Flag', 'Spd of Max Gust (km/h)', 'Spd of Max Gust Flag']


In [12]:
weather = weather_raw[
    ["Date/Time", "Mean Temp (°C)", "Total Precip (mm)"]
].copy()

weather = weather.rename(columns={
    "Date/Time": "weather_date",
    "Mean Temp (°C)": "temperature",
    "Total Precip (mm)": "precipitation"
})

weather["weather_date"] = pd.to_datetime(
    weather["weather_date"]
).dt.normalize()

for col in ["temperature", "precipitation"]:
    weather[col] = pd.to_numeric(
        weather[col],
        errors="coerce"
    )

assert weather["weather_date"].notna().all()
assert weather["weather_date"].is_unique

print("Station:", weather_raw["Station Name"].unique())
print("Climate ID:", weather_raw["Climate ID"].unique())
print("Missing weather values:")
print(weather[["temperature", "precipitation"]].isna().sum())

Station: <StringArray>
['TORONTO INTL A']
Length: 1, dtype: str
Climate ID: [6158731]
Missing weather values:
temperature      3
precipitation    2
dtype: int64


In [13]:
# Date of the historical weather observation
df["weather_date"] = (
    df["prediction_date"] - pd.Timedelta(days=1)
)

# Rename weather features
weather_features = weather.rename(columns={
    "temperature": "temperature_prev_day",
    "precipitation": "precipitation_prev_day"
})

# Join weather observations
df = df.merge(
    weather_features,
    on="weather_date",
    how="left",
    validate="many_to_one"
)

# Missing weather indicator
df["weather_missing"] = (
    df[
        ["temperature_prev_day", "precipitation_prev_day"]
    ]
    .isna()
    .any(axis=1)
    .astype("int8")
)

assert len(df) == 823_217

print("Dataset shape:", df.shape)
print("Weather missing rows:", df["weather_missing"].sum())

Dataset shape: (823217, 16)
Weather missing rows: 6767


In [14]:
output_path = "../data/processed/intersection_day_2018_features.csv"

df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")
print(f"Dataset shape: {df.shape}")

Saved to: ../data/processed/intersection_day_2018_features.csv
Dataset shape: (823217, 16)


In [15]:
feature_columns = [
    # Calendar
    "day_of_week",
    "month",
    "is_weekend",

    # Traffic
    "historical_traffic_volume",
    "traffic_missing",

    # Weather
    "temperature_prev_day",
    "precipitation_prev_day",
    "weather_missing",

    # History
    "historical_ksi_count",
    "ksi_count_30d",
    "ksi_count_365d"
]

X = df[feature_columns].copy()
y = df["target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())

X shape: (823217, 11)
y shape: (823217,)
Features: ['day_of_week', 'month', 'is_weekend', 'historical_traffic_volume', 'traffic_missing', 'temperature_prev_day', 'precipitation_prev_day', 'weather_missing', 'historical_ksi_count', 'ksi_count_30d', 'ksi_count_365d']


In [16]:
# Temporal split boundaries
train_end = pd.Timestamp("2018-09-29")
val_start = pd.Timestamp("2018-10-01")

# Date-based masks
train_mask = df["prediction_date"] <= train_end
val_mask = df["prediction_date"] >= val_start

# Training data
X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

# Validation data
X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (613223, 11)
y_train: (613223,)
X_val: (207736, 11)
y_val: (207736,)


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [18]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score


# 1. Feature groups

numeric_features = [
    "historical_traffic_volume",
    "temperature_prev_day",
    "precipitation_prev_day",
    "historical_ksi_count",
    "ksi_count_30d",
    "ksi_count_365d"
]

categorical_features = [
    "day_of_week",
    "month"
]

binary_features = [
    "is_weekend",
    "traffic_missing",
    "weather_missing"
]


# 2. Numerical preprocessing

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# 3. Categorical preprocessing

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# 4. ColumnTransformer

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
    ("binary", "passthrough", binary_features)
])


# 5. Complete ML Pipeline

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=1000
    ))
])


# 6. Model training

model.fit(X_train, y_train)


# 7. Validation probabilities

y_val_proba = model.predict_proba(X_val)[:, 1]


# 8. Evaluation

val_ap = average_precision_score(
    y_val,
    y_val_proba
)

print(f"Full model validation AP: {val_ap:.6f}")

Full model validation AP: 0.000389


In [20]:
from sklearn.base import clone

# 1. Define feature groups

feature_groups = {
    "calendar": [
        "day_of_week",
        "month",
        "is_weekend"
    ],
    "traffic": [
        "historical_traffic_volume",
        "traffic_missing"
    ],
    "weather": [
        "temperature_prev_day",
        "precipitation_prev_day",
        "weather_missing"
    ],
    "history": [
        "historical_ksi_count",
        "ksi_count_30d",
        "ksi_count_365d"
    ]
}

# 2. Define experiments

experiments = {
    "full": None,
    "without_calendar": "calendar",
    "without_traffic": "traffic",
    "without_weather": "weather",
    "without_history": "history"
}

results = []

# 3. Run ablation experiments

for experiment_name, excluded_group in experiments.items():

    selected_features = [
        col for col in feature_columns
        if excluded_group is None
        or col not in feature_groups[excluded_group]
    ]

    selected_numeric = [
        col for col in numeric_features
        if col in selected_features
    ]

    selected_categorical = [
        col for col in categorical_features
        if col in selected_features
    ]

    selected_binary = [
        col for col in binary_features
        if col in selected_features
    ]

    # Build preprocessing for available features
    transformers = []

    if selected_numeric:
        transformers.append((
            "numeric",
            clone(numeric_transformer),
            selected_numeric
        ))

    if selected_categorical:
        transformers.append((
            "categorical",
            clone(categorical_transformer),
            selected_categorical
        ))

    if selected_binary:
        transformers.append((
            "binary",
            "passthrough",
            selected_binary
        ))

    # Build a fresh pipeline
    experiment_model = Pipeline([
        ("preprocessor", ColumnTransformer(transformers)),
        ("classifier", LogisticRegression(
            C=0.1,
            max_iter=1000
        ))
    ])

    # Train and evaluate
    experiment_model.fit(
        X_train[selected_features],
        y_train
    )

    predictions = experiment_model.predict_proba(
        X_val[selected_features]
    )[:, 1]

    ap = average_precision_score(
        y_val,
        predictions
    )

    results.append({
        "experiment": experiment_name,
        "n_features": len(selected_features),
        "AP": ap
    })

# 4. Compare results

ablation_results = pd.DataFrame(results)

full_ap = ablation_results.loc[
    ablation_results["experiment"] == "full",
    "AP"
].iloc[0]

ablation_results["AP_loss_when_removed"] = (
    full_ap - ablation_results["AP"]
)

print(
    ablation_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

      experiment  n_features       AP  AP_loss_when_removed
            full          11 0.000389              0.000000
without_calendar           8 0.000326              0.000063
 without_traffic           9 0.000332              0.000057
 without_weather           8 0.000396             -0.000007
 without_history           8 0.000493             -0.000104


In [21]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone


# 1. Prepare chronological training dates

train_dates = df.loc[X_train.index, "prediction_date"]

unique_dates = (
    train_dates
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

tscv = TimeSeriesSplit(
    n_splits=3,
    gap=1
)


# 2. Define the two model configurations

cv_experiments = {
    "full": feature_columns,
    "without_history": [
        col for col in feature_columns
        if col not in feature_groups["history"]
    ]
}

cv_results = []


# 3. Temporal cross-validation

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(unique_dates),
    start=1
):

    fold_train_dates = unique_dates[train_idx]
    fold_val_dates = unique_dates[val_idx]

    train_rows = train_dates.isin(fold_train_dates)
    val_rows = train_dates.isin(fold_val_dates)

    for experiment_name, selected_features in cv_experiments.items():

        # Select columns for the current experiment

        selected_numeric = [
            col for col in numeric_features
            if col in selected_features
        ]

        selected_categorical = [
            col for col in categorical_features
            if col in selected_features
        ]

        selected_binary = [
            col for col in binary_features
            if col in selected_features
        ]

        # Build an independent preprocessing pipeline

        fold_preprocessor = ColumnTransformer([
            ("numeric", clone(numeric_transformer), selected_numeric),
            ("categorical", clone(categorical_transformer), selected_categorical),
            ("binary", "passthrough", selected_binary)
        ])

        fold_model = Pipeline([
            ("preprocessor", fold_preprocessor),
            ("classifier", LogisticRegression(
                C=0.1,
                max_iter=1000
            ))
        ])

        # Fit exclusively on the current training fold

        fold_model.fit(
            X_train.loc[train_rows, selected_features],
            y_train.loc[train_rows]
        )

        # Predict probabilities for the validation fold

        fold_proba = fold_model.predict_proba(
            X_train.loc[val_rows, selected_features]
        )[:, 1]

        fold_ap = average_precision_score(
            y_train.loc[val_rows],
            fold_proba
        )

        cv_results.append({
            "fold": fold,
            "experiment": experiment_name,
            "AP": fold_ap
        })


# 4. Aggregate results

cv_results_df = pd.DataFrame(cv_results)

cv_summary = cv_results_df.pivot(
    index="fold",
    columns="experiment",
    values="AP"
)

cv_summary = cv_summary[
    ["full", "without_history"]
]

cv_summary["difference"] = (
    cv_summary["without_history"] - cv_summary["full"]
)

cv_summary.loc["Mean"] = cv_summary.mean()

print(
    cv_summary.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

experiment     full  without_history  difference
fold                                            
1          0.000243         0.000200   -0.000044
2          0.000522         0.000530    0.000008
3          0.002162         0.000722   -0.001440
Mean       0.000976         0.000484   -0.000492


## Feature Engineering — Experimental Conclusions

### Objective

Evaluate leakage-aware calendar, traffic, weather,
and historical KSI features for next-day accident
risk ranking at Toronto signalized intersections.

### Dataset

- Observations: 823,217
- Prediction unit: intersection × prediction day
- Target: matched KSI event on the next day
- Feature groups: Calendar, Traffic, Weather, History
- Model: LogisticRegression(C=0.1)
- Primary metric: Average Precision

### Development Validation

- Full model AP: 0.000389
- Without Calendar: 0.000326
- Without Traffic: 0.000332
- Without Weather: 0.000396
- Without History: 0.000493

### Temporal Cross-Validation

Mean AP across three temporal folds:

- Full model: 0.000976
- Without History: 0.000484

### Findings

1. Calendar and Traffic features contributed positively
   on the development validation period.

2. Weather features showed no clear improvement.

3. Removing History improved development validation AP,
   but this advantage did not generalize consistently
   across temporal CV folds.

4. The results do not establish a stable improvement
   over the previous baseline.

### Limitations

- Extremely rare positive events.
- Limited matched KSI outcome coverage.
- Incomplete historical traffic measurements.
- Historical availability of external data is not
  fully verified.
- Weather observations come from one station.
- Final untouched test evaluation has not been performed.

### Conclusion

The feature engineering workflow and ablation experiments
provide preliminary evidence about feature usefulness.

No feature group is established as universally beneficial
or harmful.

Further model selection requires broader temporal
validation and an untouched final test period.